# Machine Learning Programming Quiz Reviewer Notebook

**Purpose:** One practical, syntax-focused notebook for the topics covered by the course notebooks in this repository.

Use this as a quiz survival notebook: copy a template, replace the dataset/target names, run, then interpret the metrics.

## Reference notebooks scanned
- Module 1: moving average, least squares regression, dummy variables, other model types
- Module 2: bike rental demand regression, normal equation, regression metrics, regularization
- Model Evaluation: binary classification metrics, SVM, ROC/PR curves, threshold calibration, cost-sensitive decisions
- Module 4: CART/Decision Trees, Gini/entropy, pruning, hyperparameter search, nested CV
- Module 5–6: ANN forward/backward pass, NumPy neural nets, PyTorch training loops
- Module 6 CNN: image augmentation, pretrained CNNs, transfer learning/fine-tuning
- Notes: CART discussion, PyTorch tutorial, Transformers


## 0. Quiz mindset: standard ML workflow

1. **Load** data with `pd.read_csv()` or sklearn/OpenML loader.
2. **Inspect** shape, columns, missing values, target distribution.
3. **Split** before fitting scalers/encoders to avoid leakage.
4. **Preprocess** numeric/categorical features.
5. **Train** baseline model first.
6. **Evaluate** using correct metrics.
7. **Tune** hyperparameters with cross-validation.
8. **Interpret** coefficients, feature importances, confusion matrix, curves.

Common mistakes in programming quizzes:
- Scaling the full dataset before split.
- Forgetting `stratify=y` for classification.
- Using accuracy only on imbalanced data.
- Using `predict()` scores for ROC-AUC instead of probabilities/decision scores.
- Forgetting intercept/bias column in normal equation.
- One-hot encoding train/test separately instead of using a pipeline or aligning columns.


In [ ]:
# Core imports used throughout the course notebooks
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.datasets import make_regression, make_classification, make_moons, load_iris, load_wine

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)


## 1. Pandas / NumPy essentials

Use these for almost every quiz problem.


In [ ]:
# Load a CSV
# df = pd.read_csv('path/to/file.csv')

# Demo dataframe
rng = np.random.default_rng(42)
df = pd.DataFrame({
    'x1': rng.normal(size=8),
    'x2': rng.integers(1, 5, size=8),
    'category': ['A', 'B', 'A', 'C', 'B', 'C', 'A', 'B'],
    'target': rng.normal(10, 2, size=8)
})

display(df.head())
print('shape:', df.shape)
print('\nInfo:')
df.info()
print('\nMissing values:')
print(df.isna().sum())
print('\nSummary:')
display(df.describe(include='all'))


In [ ]:
# Feature/target split
X = df.drop(columns='target')
y = df['target']

# Quick categorical encoding with pandas
X_dummies = pd.get_dummies(X, columns=['category'], drop_first=True)
display(X_dummies)

# Train/test split syntax
X_train, X_test, y_train, y_test = train_test_split(
    X_dummies, y, test_size=0.2, random_state=RANDOM_STATE
)


## 2. Moving average / rolling windows

**Use case:** smooth noisy time-series data such as weather, stock prices, or daily counts.

Key syntax: `series.rolling(window=20).mean()`.


In [ ]:
# Moving average with synthetic data
n = 120
date_index = pd.date_range('2024-01-01', periods=n, freq='D')
values = 20 + np.sin(np.arange(n) / 8) * 5 + np.random.normal(0, 1.5, n)
ts = pd.DataFrame({'date': date_index, 'temperature': values})
ts['ma_7'] = ts['temperature'].rolling(window=7).mean()
ts['ma_20'] = ts['temperature'].rolling(window=20).mean()

ts.plot(x='date', y=['temperature', 'ma_7', 'ma_20'], alpha=0.85)
plt.title('Moving Average Smoothing')
plt.ylabel('value')
plt.show()


In [ ]:
# If asked to implement rolling average manually
window = 5
manual_ma = []
for i in range(len(ts)):
    if i + 1 < window:
        manual_ma.append(np.nan)
    else:
        manual_ma.append(ts['temperature'].iloc[i-window+1:i+1].mean())

ts['manual_ma_5'] = manual_ma
display(ts[['temperature', 'manual_ma_5']].head(10))


## 3. Least squares linear regression

### Formula
For feature matrix `X` with bias column and target `y`:

$\theta = (X^T X)^{-1}X^T y$

In code, prefer `np.linalg.pinv(X)` over inverse because it is safer for singular matrices.


In [ ]:
# Generate regression data
X_np, y_np = make_regression(n_samples=120, n_features=1, noise=15, random_state=42)

# Normal equation from scratch
X_bias = np.c_[np.ones(X_np.shape[0]), X_np]  # add intercept column
# theta = np.linalg.inv(X_bias.T @ X_bias) @ X_bias.T @ y_np  # classic formula
# Safer equivalent:
theta = np.linalg.pinv(X_bias) @ y_np
print('intercept, slope:', theta)

# Predict
y_pred_manual = X_bias @ theta

# Compare with sklearn
lr = LinearRegression()
lr.fit(X_np, y_np)
y_pred_sklearn = lr.predict(X_np)
print('sklearn intercept:', lr.intercept_)
print('sklearn coef:', lr.coef_)
print('R2:', r2_score(y_np, y_pred_sklearn))

plt.scatter(X_np[:, 0], y_np, alpha=0.6)
order = np.argsort(X_np[:, 0])
plt.plot(X_np[order, 0], y_pred_manual[order], color='red', label='manual normal equation')
plt.legend(); plt.title('Least Squares Regression'); plt.show()


In [ ]:
# Regression metrics template

def regression_report(y_true, y_pred, name='model'):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f'[{name}]')
    print('MAE :', mean_absolute_error(y_true, y_pred))
    print('MSE :', mse)
    print('RMSE:', rmse)
    print('R2  :', r2_score(y_true, y_pred))

regression_report(y_np, y_pred_sklearn, 'LinearRegression')


## 4. Dummy variables / categorical features

**Use case:** convert text categories (`season`, `weather`, `city`) into numeric columns.

Rules:
- Use `drop_first=True` in simple linear regression to avoid the dummy variable trap.
- With sklearn pipelines, `OneHotEncoder(handle_unknown='ignore')` is safer for test sets.


In [ ]:
# Pandas get_dummies approach
cat_df = pd.DataFrame({
    'size': [30, 40, 45, 60, 70, 80],
    'location': ['urban', 'rural', 'suburban', 'urban', 'rural', 'suburban'],
    'price': [120, 90, 110, 170, 140, 160]
})

encoded = pd.get_dummies(cat_df, columns=['location'], drop_first=True)
X = encoded.drop(columns='price')
y = encoded['price']
model = LinearRegression().fit(X, y)
print(pd.Series(model.coef_, index=X.columns))
print('Intercept:', model.intercept_)


In [ ]:
# Pipeline approach: best for quizzes with mixed numeric/categorical data
mixed_df = pd.DataFrame({
    'temp': [25, 28, 21, 30, 18, 20, 27, 29],
    'humidity': [70, 60, 80, 55, 90, 85, 65, 58],
    'season': ['summer', 'summer', 'winter', 'summer', 'winter', 'winter', 'spring', 'spring'],
    'rentals': [300, 350, 180, 390, 120, 150, 270, 310]
})

X = mixed_df.drop(columns='rentals')
y = mixed_df['rentals']
num_cols = ['temp', 'humidity']
cat_cols = ['season']

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

pipe = Pipeline([
    ('prep', preprocess),
    ('model', LinearRegression())
])
pipe.fit(X, y)
print('Training R2:', pipe.score(X, y))


## 5. Polynomial regression and other regression models

**Use when:** linear model underfits curved data.

Common regressors in the course:
- `LinearRegression()` baseline.
- `Ridge(alpha=...)` L2 regularization, keeps all features but shrinks coefficients.
- `Lasso(alpha=...)` L1 regularization, can force coefficients to zero.
- `ElasticNet(alpha=..., l1_ratio=...)` mix of Ridge and Lasso.
- `KNeighborsRegressor(n_neighbors=...)` non-parametric local averaging.


In [ ]:
# Polynomial regression with Pipeline
x = np.linspace(-3, 3, 150).reshape(-1, 1)
y = 2 * x[:, 0]**2 - 3 * x[:, 0] + 5 + np.random.normal(0, 2, size=len(x))

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('linear', LinearRegression())
])
poly_model.fit(X_train, y_train)
y_pred = poly_model.predict(X_test)
regression_report(y_test, y_pred, 'Polynomial degree=2')

xx = np.linspace(-3, 3, 300).reshape(-1, 1)
plt.scatter(X_train[:, 0], y_train, alpha=0.4, label='train')
plt.scatter(X_test[:, 0], y_test, alpha=0.4, label='test')
plt.plot(xx[:, 0], poly_model.predict(xx), color='red', label='poly fit')
plt.legend(); plt.show()


In [ ]:
# Baseline comparison with cross_validate
X_reg, y_reg = make_regression(n_samples=300, n_features=15, noise=25, random_state=42)
models = {
    'linear': LinearRegression(),
    'ridge': Ridge(alpha=1.0),
    'lasso': Lasso(alpha=0.1, max_iter=10000),
    'elastic': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
    'knn': KNeighborsRegressor(n_neighbors=5)
}

rows = []
for name, model in models.items():
    scores = cross_validate(model, X_reg, y_reg, cv=5, scoring=('r2', 'neg_root_mean_squared_error'))
    rows.append({
        'model': name,
        'mean_r2': scores['test_r2'].mean(),
        'mean_rmse': -scores['test_neg_root_mean_squared_error'].mean()
    })

pd.DataFrame(rows).sort_values('mean_r2', ascending=False)


In [ ]:
# Regularization hyperparameter tuning
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
}
search = GridSearchCV(Ridge(), param_grid=param_grid, cv=5, scoring='r2')
search.fit(X_reg, y_reg)
print('Best alpha:', search.best_params_)
print('Best CV R2:', search.best_score_)


## 6. Bike rental demand / full regression workflow

This mirrors the bike demand notebook: load CSV, select target, preprocess, train, evaluate, sanity-check with sklearn.


In [ ]:
# Bike demand workflow template using local day.csv if available
from pathlib import Path

bike_path = Path('[2] Group 8 - Module 2 - Predicting Bike Rental Demand with Linear Regression/day.csv')
if bike_path.exists():
    bike = pd.read_csv(bike_path)
    print('Loaded:', bike_path)
else:
    # fallback toy data if file is unavailable
    bike = pd.DataFrame({
        'temp': np.random.rand(200),
        'hum': np.random.rand(200),
        'windspeed': np.random.rand(200),
        'season': np.random.choice([1, 2, 3, 4], 200),
        'workingday': np.random.choice([0, 1], 200),
        'cnt': np.random.randint(100, 8000, 200)
    })

print(bike.shape)
display(bike.head())
print(bike.columns.tolist())


In [ ]:
# Typical Bike Sharing columns: target is 'cnt'; avoid leakage columns 'casual' and 'registered'
target = 'cnt'
leakage_or_id_cols = [c for c in ['instant', 'dteday', 'casual', 'registered'] if c in bike.columns]
X = bike.drop(columns=[target] + leakage_or_id_cols)
y = bike[target]

# Treat low-cardinality integer variables as categorical if desired
cat_cols = [c for c in X.columns if X[c].dtype == 'object']
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg_pipe = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])),
    ('model', LinearRegression())
])

reg_pipe.fit(X_train, y_train)
y_pred = reg_pipe.predict(X_test)
regression_report(y_test, y_pred, 'Bike demand LinearRegression')


## 7. Classification metrics and SVM model evaluation

For binary classification:
- **Accuracy:** overall correct predictions.
- **Precision:** among predicted positives, how many are truly positive? Use when false positives are costly.
- **Recall/Sensitivity:** among actual positives, how many were found? Use when false negatives are costly.
- **F1:** harmonic mean of precision and recall.
- **ROC-AUC:** ranking quality over thresholds.
- **Confusion matrix:** `[[TN, FP], [FN, TP]]`.


In [ ]:
# Binary classification data
X_cls, y_cls = make_classification(
    n_samples=600, n_features=12, n_informative=5, n_redundant=2,
    weights=[0.65, 0.35], random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=42, stratify=y_cls
)

# SVM with scaling is important
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
])
svm_pipe.fit(X_train, y_train)

y_pred = svm_pipe.predict(X_test)
y_score = svm_pipe.predict_proba(X_test)[:, 1]  # or decision_function if probability=False

print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_score))
cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1']))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()


In [ ]:
# ROC curve and Precision-Recall curve
fpr, tpr, roc_thresholds = roc_curve(y_test, y_score)
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_score)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f'AUC={roc_auc_score(y_test, y_score):.3f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate / Recall')
plt.title('ROC Curve'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(recall, precision)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.tight_layout(); plt.show()


In [ ]:
# Threshold calibration template
# Default classifier threshold is usually 0.5. Change it based on costs or desired recall/precision.

def predict_with_threshold(scores, threshold):
    return (scores >= threshold).astype(int)

for threshold in [0.3, 0.5, 0.7]:
    pred_t = predict_with_threshold(y_score, threshold)
    print('\nThreshold:', threshold)
    print('precision:', precision_score(y_test, pred_t, zero_division=0))
    print('recall   :', recall_score(y_test, pred_t, zero_division=0))
    print('f1       :', f1_score(y_test, pred_t, zero_division=0))
    print('cm       :\n', confusion_matrix(y_test, pred_t))


In [ ]:
# Cost-sensitive threshold selection
# Example: false negative is 5x more costly than false positive.
FP_COST = 1
FN_COST = 5
thresholds = np.linspace(0.01, 0.99, 99)

records = []
for t in thresholds:
    pred_t = predict_with_threshold(y_score, t)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    total_cost = FP_COST * fp + FN_COST * fn
    records.append((t, fp, fn, total_cost))

cost_df = pd.DataFrame(records, columns=['threshold', 'FP', 'FN', 'cost'])
best = cost_df.loc[cost_df['cost'].idxmin()]
print(best)

plt.plot(cost_df['threshold'], cost_df['cost'])
plt.axvline(best['threshold'], color='red', linestyle='--')
plt.xlabel('Threshold'); plt.ylabel('Total cost'); plt.title('Cost vs Threshold')
plt.show()


In [ ]:
# RandomizedSearchCV for SVM hyperparameters
from scipy.stats import loguniform

param_dist = {
    'model__C': loguniform(1e-2, 1e2),
    'model__gamma': loguniform(1e-4, 1e0)
}

svm_search = RandomizedSearchCV(
    svm_pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1
)
svm_search.fit(X_train, y_train)
print('Best params:', svm_search.best_params_)
print('Best CV ROC-AUC:', svm_search.best_score_)


## 8. CART / Decision Trees

CART = Classification and Regression Trees.

Important ideas:
- Tree recursively chooses splits that reduce impurity.
- Classification criteria: **Gini** or **entropy**.
- Regression criterion: squared error / variance reduction.
- Deep trees overfit; control with `max_depth`, `min_samples_split`, `min_samples_leaf`, or pruning `ccp_alpha`.


In [ ]:
# Gini and entropy from scratch

def gini(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p**2)


def entropy(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p + 1e-12))


def weighted_impurity(y_left, y_right, criterion=gini):
    n = len(y_left) + len(y_right)
    return (len(y_left) / n) * criterion(y_left) + (len(y_right) / n) * criterion(y_right)

sample_y = np.array([0, 0, 0, 1, 1, 1, 1])
print('gini:', gini(sample_y))
print('entropy:', entropy(sample_y))
print('weighted split impurity:', weighted_impurity(sample_y[:3], sample_y[3:], gini))


In [ ]:
# Best split skeleton from scratch

def best_split(X, y, criterion=gini):
    best = {'feature': None, 'threshold': None, 'impurity': np.inf}
    n_features = X.shape[1]
    for j in range(n_features):
        thresholds = np.unique(X[:, j])
        for t in thresholds:
            left = X[:, j] <= t
            right = ~left
            if left.sum() == 0 or right.sum() == 0:
                continue
            impurity = weighted_impurity(y[left], y[right], criterion)
            if impurity < best['impurity']:
                best = {'feature': j, 'threshold': t, 'impurity': impurity}
    return best

X_demo, y_demo = make_moons(n_samples=100, noise=0.2, random_state=42)
print(best_split(X_demo, y_demo, gini))


In [ ]:
# sklearn DecisionTreeClassifier
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=42, stratify=iris.target
)

tree_clf = DecisionTreeClassifier(
    criterion='gini', max_depth=3, random_state=42
)
tree_clf.fit(X_train, y_train)
y_pred = tree_clf.predict(X_test)
print('accuracy:', accuracy_score(y_test, y_pred))
print(export_text(tree_clf, feature_names=list(iris.feature_names)))

plt.figure(figsize=(12, 6))
plot_tree(tree_clf, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.show()


In [ ]:
# DecisionTreeRegressor example
Xr, yr = make_regression(n_samples=250, n_features=5, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(Xr, yr, test_size=0.25, random_state=42)

tree_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_reg.fit(X_train, y_train)
regression_report(y_test, tree_reg.predict(X_test), 'DecisionTreeRegressor')


In [ ]:
# Cost-complexity pruning with ccp_alpha
path = tree_clf.cost_complexity_pruning_path(iris.data, iris.target)
ccp_alphas = path.ccp_alphas
print('first alphas:', ccp_alphas[:5])

param_grid = {'ccp_alpha': ccp_alphas, 'max_depth': [2, 3, 4, None]}
prune_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=5
)
prune_search.fit(iris.data, iris.target)
print('Best params:', prune_search.best_params_)
print('Best CV accuracy:', prune_search.best_score_)


In [ ]:
# Nested CV pattern: inner CV tunes, outer CV estimates generalization
inner = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={'max_depth': [2, 3, 4, 5, None], 'min_samples_leaf': [1, 2, 5]},
    cv=3,
    scoring='accuracy'
)
outer_scores = cross_val_score(inner, iris.data, iris.target, cv=5, scoring='accuracy')
print('Nested CV scores:', outer_scores)
print('Mean:', outer_scores.mean(), 'Std:', outer_scores.std())


## 9. NumPy ANN: one-hot, softmax, forward pass, backprop

This mirrors the ANN contest notebooks. Know the shapes.

Shape convention below:
- `X`: `(n_samples, n_features)`
- `y_onehot`: `(n_samples, n_classes)`
- Layer 1 weights `W1`: `(n_features, hidden_units)`
- Layer 2 weights `W2`: `(hidden_units, n_classes)`


In [ ]:
# Basic neural network helper functions in NumPy

def one_hot(y, num_classes=None):
    y = np.asarray(y, dtype=int)
    if num_classes is None:
        num_classes = y.max() + 1
    out = np.zeros((len(y), num_classes))
    out[np.arange(len(y)), y] = 1
    return out


def relu(z):
    return np.maximum(0, z)


def relu_grad(z):
    return (z > 0).astype(float)


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)  # stability
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=1, keepdims=True)


def cross_entropy(probs, y_onehot):
    eps = 1e-12
    return -np.mean(np.sum(y_onehot * np.log(probs + eps), axis=1))


In [ ]:
# Two-layer neural network from scratch
X, y = make_classification(n_samples=300, n_features=4, n_informative=4, n_redundant=0,
                           n_classes=3, random_state=42)
X = StandardScaler().fit_transform(X)
Y = one_hot(y, num_classes=3)

n, d = X.shape
h = 8
k = Y.shape[1]
rng = np.random.default_rng(42)
params = {
    'W1': rng.normal(0, 0.1, size=(d, h)),
    'b1': np.zeros((1, h)),
    'W2': rng.normal(0, 0.1, size=(h, k)),
    'b2': np.zeros((1, k))
}

def forward(X, params):
    z1 = X @ params['W1'] + params['b1']
    a1 = relu(z1)
    logits = a1 @ params['W2'] + params['b2']
    probs = softmax(logits)
    cache = {'z1': z1, 'a1': a1, 'logits': logits, 'probs': probs}
    return probs, cache

def backward(X, Y, params, cache):
    n = X.shape[0]
    dz2 = (cache['probs'] - Y) / n
    dW2 = cache['a1'].T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)
    da1 = dz2 @ params['W2'].T
    dz1 = da1 * relu_grad(cache['z1'])
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

def update(params, grads, lr=0.1):
    for key in params:
        params[key] -= lr * grads[key]

for epoch in range(300):
    probs, cache = forward(X, params)
    loss = cross_entropy(probs, Y)
    grads = backward(X, Y, params, cache)
    update(params, grads, lr=0.1)
    if epoch % 100 == 0:
        pred = probs.argmax(axis=1)
        print(epoch, 'loss=', round(loss, 4), 'acc=', round(accuracy_score(y, pred), 4))

pred = forward(X, params)[0].argmax(axis=1)
print('Final accuracy:', accuracy_score(y, pred))


In [ ]:
# Tiny gradient check pattern for one parameter
# Numerical gradient ~= analytical gradient

def loss_given_params(params):
    probs, _ = forward(X, params)
    return cross_entropy(probs, Y)

probs, cache = forward(X, params)
grads = backward(X, Y, params, cache)

epsilon = 1e-5
i, j = 0, 0
original = params['W1'][i, j]
params['W1'][i, j] = original + epsilon
loss_plus = loss_given_params(params)
params['W1'][i, j] = original - epsilon
loss_minus = loss_given_params(params)
params['W1'][i, j] = original

numerical = (loss_plus - loss_minus) / (2 * epsilon)
analytical = grads['W1'][i, j]
print('numerical:', numerical)
print('analytical:', analytical)
print('absolute difference:', abs(numerical - analytical))


## 10. PyTorch essentials

Know these objects:
- `torch.tensor`, tensor shapes, `.to(device)`
- `TensorDataset`, `DataLoader`
- `nn.Module`, `forward()`
- optimizer: `torch.optim.Adam` or `SGD`
- loss: `nn.CrossEntropyLoss()` for multiclass logits, `nn.MSELoss()` for regression


In [ ]:
# PyTorch classifier template. Run only if torch is installed.
try:
    import torch
    from torch import nn
    from torch.utils.data import TensorDataset, DataLoader

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('device:', device)

    X_pt, y_pt = make_classification(n_samples=500, n_features=10, n_classes=2, random_state=42)
    X_pt = StandardScaler().fit_transform(X_pt).astype('float32')
    y_pt = y_pt.astype('int64')

    X_train, X_test, y_train, y_test = train_test_split(X_pt, y_pt, test_size=0.2, stratify=y_pt, random_state=42)
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128)

    class MLP(nn.Module):
        def __init__(self, in_features, hidden=32, out_features=2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_features, hidden),
                nn.ReLU(),
                nn.Linear(hidden, out_features)
            )
        def forward(self, x):
            return self.net(x)

    model = MLP(in_features=X_train.shape[1]).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(5):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)
        print(f'epoch {epoch}: train loss={total_loss / len(train_ds):.4f}')

    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            logits = model(xb.to(device))
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
    print('test accuracy:', accuracy_score(y_test, all_preds))

except ImportError:
    print('torch is not installed in this environment. Use this cell as syntax reference.')


## 11. CNN image classification and data augmentation with Keras/TensorFlow

Important concepts from the CNN notebooks:
- `ImageDataGenerator` loads images from directory structure: `train/cats`, `train/dogs`, etc.
- Data augmentation reduces overfitting: rotation, shifts, shear, zoom, horizontal flip.
- Pretrained CNNs (`VGG16`, `VGG19`, `Xception`) can be used as frozen feature extractors.
- Fine-tuning unfreezes later convolutional layers with a small learning rate.


In [ ]:
# Keras ImageDataGenerator syntax reference. Run only if tensorflow is installed and directories exist.
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models, optimizers
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.applications import VGG16, VGG19, Xception

    IMG_SIZE = (150, 150)
    BATCH_SIZE = 32

    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=40,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    val_datagen = ImageDataGenerator(rescale=1./255)

    # Directory format:
    # data/train/class_a/*.jpg
    # data/train/class_b/*.jpg
    # data/validation/class_a/*.jpg
    # data/validation/class_b/*.jpg
    # train_gen = train_datagen.flow_from_directory('data/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary')
    # val_gen = val_datagen.flow_from_directory('data/validation', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary')

    cnn = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    cnn.compile(optimizer=optimizers.RMSprop(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    cnn.summary()

    # history = cnn.fit(train_gen, epochs=20, validation_data=val_gen,
    #                   callbacks=[EarlyStopping(patience=3, restore_best_weights=True)])
except ImportError:
    print('tensorflow is not installed. Use this cell as syntax reference.')


In [ ]:
# Transfer learning template with VGG16/VGG19/Xception
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models, optimizers
    from tensorflow.keras.applications import VGG16, VGG19, Xception

    conv_base = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
    conv_base.trainable = False  # freeze pretrained convolutional base

    model = models.Sequential([
        conv_base,
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizers.RMSprop(learning_rate=1e-4),
                  loss='binary_crossentropy', metrics=['accuracy'])

    # Fine-tuning pattern: unfreeze top layers with very small LR
    # conv_base.trainable = True
    # for layer in conv_base.layers[:-4]:
    #     layer.trainable = False
    # model.compile(optimizer=optimizers.RMSprop(learning_rate=1e-5),
    #               loss='binary_crossentropy', metrics=['accuracy'])

    print('Transfer learning model ready.')
except ImportError:
    print('tensorflow is not installed. Use this cell as syntax reference.')


## 12. Transformer essentials

Transformer quiz concepts:
- Token ids go through `nn.Embedding`.
- Add positional information because self-attention has no natural order.
- `nn.Transformer` can do sequence-to-sequence tasks.
- Output logits shape often needs reshaping before `CrossEntropyLoss`.


In [ ]:
# Minimal PyTorch Transformer syntax reference for sequence-to-sequence tasks.
try:
    import math
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader

    class ReverseDataset(Dataset):
        def __init__(self, vocab_size=20, seq_len=6, size=256):
            self.data = torch.randint(1, vocab_size, (size, seq_len))
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            x = self.data[idx]
            y = torch.flip(x, dims=(0,))
            return x, y

    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=500):
            super().__init__()
            pe = torch.zeros(max_len, d_model)
            pos = torch.arange(0, max_len).unsqueeze(1)
            div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(pos * div)
            pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer('pe', pe.unsqueeze(0))
        def forward(self, x):
            return x + self.pe[:, :x.size(1)]

    class TinyTransformer(nn.Module):
        def __init__(self, vocab_size=20, d_model=32, nhead=4, num_layers=1):
            super().__init__()
            self.embed = nn.Embedding(vocab_size, d_model)
            self.pos = PositionalEncoding(d_model)
            self.transformer = nn.Transformer(
                d_model=d_model, nhead=nhead,
                num_encoder_layers=num_layers, num_decoder_layers=num_layers,
                batch_first=True
            )
            self.out = nn.Linear(d_model, vocab_size)
        def forward(self, src, tgt):
            src_emb = self.pos(self.embed(src))
            tgt_emb = self.pos(self.embed(tgt))
            hidden = self.transformer(src_emb, tgt_emb)
            return self.out(hidden)  # (batch, seq_len, vocab_size)

    ds = ReverseDataset()
    xb, yb = next(iter(DataLoader(ds, batch_size=8)))
    model = TinyTransformer()
    logits = model(xb, yb)  # teacher forcing example
    loss = nn.CrossEntropyLoss()(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
    print('logits shape:', logits.shape, 'loss:', loss.item())
except ImportError:
    print('torch is not installed. Use this cell as syntax reference.')


## 13. Fast templates to memorize

### Regression template
```python
X = df.drop(columns='target')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(mean_absolute_error(y_test, pred), np.sqrt(mean_squared_error(y_test, pred)), r2_score(y_test, pred))
```

### Classification template
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
model = Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
score = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred))
print(roc_auc_score(y_test, score))
```

### Tree template
```python
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
plot_tree(tree, filled=True)
print(export_text(tree))
```

### ANN backprop mental model
1. Forward: `z = X @ W + b`, activation, logits, softmax.
2. Loss: cross-entropy.
3. Backward: gradient of softmax + CE is `(probs - y_onehot) / n`.
4. Update: `param -= learning_rate * gradient`.


## 14. Metric selection cheat sheet

| Task | Main metrics | Notes |
|---|---|---|
| Regression | MAE, MSE, RMSE, R2 | RMSE penalizes large errors more than MAE. R2 closer to 1 is better. |
| Balanced classification | Accuracy, F1 | Accuracy is okay when classes are balanced. |
| Imbalanced classification | Precision, recall, F1, ROC-AUC, PR curve | Accuracy can be misleading. |
| Screening / detection | Recall | Minimize false negatives. |
| Expensive false alarms | Precision | Minimize false positives. |
| Probabilistic ranking | ROC-AUC | Use scores/probabilities, not hard labels. |
| Cost-sensitive classification | Custom threshold/cost function | Pick threshold minimizing expected cost. |

## Last-minute checklist
- Did you identify `X` and `y` correctly?
- Did you remove ID/leakage columns?
- Did you split before fitting scalers/encoders?
- Did you use `stratify=y` for classification?
- Did you choose the correct metric?
- Did you set `random_state` for reproducibility?
